# 🏠 Análise de Preços de Imóveis — California Housing Dataset

**Autor:** Análise Exploratória e Inferencial  
**Dataset:** California Housing Dataset (20.640 registros — Censo 1990)  
**Objetivo:** Análise estatística completa, identificação de padrões, relações entre variáveis e inferências sobre os preços medianos de imóveis na Califórnia.

---

## Estrutura do Projeto

1. **Setup & Importações**
2. **Carregamento e Inspeção Inicial**
3. **Limpeza e Tratamento de Dados**
4. **Estatísticas Descritivas**
5. **Análise de Distribuições**
6. **Análise Geoespacial**
7. **Análise por Proximidade ao Oceano**
8. **Matriz de Correlação e Relações entre Variáveis**
9. **Engenharia de Features — Variáveis Derivadas**
10. **Análise de Outliers**
11. **Testes de Hipóteses e Inferência Estatística**
12. **Modelo Preditivo Baseline (Regressão Linear)**
13. **Conclusões e Insights**


## 1. Setup & Importações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import (
    shapiro, normaltest, kstest, mannwhitneyu, kruskal,
    spearmanr, pearsonr, f_oneway, levene
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import itertools
from pathlib import Path

# ── Estrutura de diretórios ───────────────────────────────────────────────────
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

def savefig(name: str, **kwargs):
    """Salva figura em figures/ com configurações padrão."""
    path = FIGURES_DIR / name
    plt.savefig(path, bbox_inches='tight', dpi=130, **kwargs)

# ── Estilo global — compatível com matplotlib 3.x ────────────────────────────
_pref = ['seaborn-v0_8-whitegrid', 'seaborn-whitegrid', 'ggplot']
for _s in _pref:
    if _s in plt.style.available:
        plt.style.use(_s)
        break

sns.set_theme(style='whitegrid', palette='muted')

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'figure.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860', '#DA8BC3']
OCEAN_COLORS = {
    'NEAR BAY':   '#2196F3',
    '<1H OCEAN':  '#4CAF50',
    'INLAND':     '#FF9800',
    'NEAR OCEAN': '#9C27B0',
    'ISLAND':     '#F44336',
}

print("✅ Bibliotecas carregadas com sucesso.")
print(f"   NumPy {np.__version__} | Pandas {pd.__version__} | Seaborn {sns.__version__}")
print(f"   Matplotlib {plt.matplotlib.__version__}")
print(f"   Figuras salvas em: {FIGURES_DIR.resolve()}")

## 2. Carregamento e Inspeção Inicial

In [ ]:
DATA_PATH = "housing.csv"
df_raw = pd.read_csv(DATA_PATH)

print("=" * 60)
print("INSPEÇÃO INICIAL DO DATASET")
print("=" * 60)
print(f"\n📐 Shape: {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas")
print("\n📋 Colunas e tipos:")
print(df_raw.dtypes.to_string())
print("\n🔍 Primeiras 5 linhas:")
df_raw.head()

In [ ]:
print("=" * 60)
print("RESUMO DE VALORES AUSENTES")
print("=" * 60)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Valores Ausentes': missing,
    '% do Total': missing_pct
}).query('`Valores Ausentes` > 0')

if missing_df.empty:
    print("\n✅ Nenhum valor ausente encontrado.")
else:
    print(missing_df.to_string())

print("\n" + "=" * 60)
print("DISTRIBUIÇÃO DA VARIÁVEL CATEGÓRICA")
print("=" * 60)
print(df_raw['ocean_proximity'].value_counts().to_string())

## 3. Limpeza e Tratamento de Dados

In [ ]:
df = df_raw.copy()

# ── 1. Imputação de valores ausentes em total_bedrooms com mediana ──────────
n_missing_before = df['total_bedrooms'].isnull().sum()
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)
print(f"✅ total_bedrooms: {n_missing_before} valores ausentes imputados com a mediana "
      f"({df_raw['total_bedrooms'].median():.0f}).")

# ── 2. Renomear colunas para leitura facilitada ──────────────────────────────
df.rename(columns={
    'longitude':          'longitude',
    'latitude':           'latitude',
    'housing_median_age': 'idade_mediana',
    'total_rooms':        'total_quartos',
    'total_bedrooms':     'total_dormitorios',
    'population':         'populacao',
    'households':         'domicilios',
    'median_income':      'renda_mediana',
    'median_house_value': 'valor_mediano',
    'ocean_proximity':    'proximidade_oceano',
}, inplace=True)

# ── 3. Verificar duplicatas ──────────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f"{'✅' if n_dup == 0 else '⚠️'} Duplicatas encontradas: {n_dup}")

# ── 4. Verificar capping artificial em valor_mediano (500k) ─────────────────
capped = (df['valor_mediano'] == 500_000).sum()
pct_capped = capped / len(df) * 100
print(f"\n⚠️  Registros com valor_mediano = $500.000 (teto artificial): "
      f"{capped:,} ({pct_capped:.1f}%) — isso representa um censoring superior nos dados.")

print(f"\n✅ Dataset limpo: {df.shape[0]:,} × {df.shape[1]} colunas")
df.head(3)

## 4. Estatísticas Descritivas

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

desc = df[num_cols].describe().T
desc['skewness'] = df[num_cols].skew().round(3)
desc['kurtosis'] = df[num_cols].kurtosis().round(3)
desc['cv_%']     = (df[num_cols].std() / df[num_cols].mean() * 100).round(1)
desc['iqr']      = (df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)).round(2)

print("=" * 60)
print("ESTATÍSTICAS DESCRITIVAS COMPLETAS")
print("=" * 60)
pd.options.display.float_format = '{:,.3f}'.format
desc.round(2)

In [ ]:
print("=" * 60)
print("INTERPRETAÇÃO — VARIÁVEL-ALVO: valor_mediano")
print("=" * 60)
vm = df['valor_mediano']
q1, q3 = vm.quantile(0.25), vm.quantile(0.75)
print(f"  Média:      ${vm.mean():>12,.0f}")
print(f"  Mediana:    ${vm.median():>12,.0f}")
print(f"  Desvio-pad: ${vm.std():>12,.0f}")
print(f"  Mínimo:     ${vm.min():>12,.0f}")
print(f"  Máximo:     ${vm.max():>12,.0f}")
print(f"  Q1 (25%):   ${q1:>12,.0f}")
print(f"  Q3 (75%):   ${q3:>12,.0f}")
print(f"  IQR:        ${(q3 - q1):>12,.0f}")
print(f"  Assimetria: {vm.skew():>12.3f}  → {'positiva (cauda longa à direita)' if vm.skew() > 0 else 'negativa'}")
print(f"  Curtose:    {vm.kurtosis():>12.3f}  → {'leptocúrtica (cauda pesada)' if vm.kurtosis() > 0 else 'platicúrtica'}")
print(f"\n  → A diferença média/mediana (${vm.mean()-vm.median():,.0f}) indica que")
print(f"    imóveis de alto valor puxam a média para cima.")

## 5. Análise de Distribuições

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

plot_cols = ['valor_mediano', 'renda_mediana', 'idade_mediana',
             'total_quartos', 'total_dormitorios', 'populacao',
             'domicilios', 'longitude', 'latitude']

for i, col in enumerate(plot_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=50, color=PALETTE[i % len(PALETTE)], alpha=0.75, edgecolor='white')
    mean_v, median_v = data.mean(), data.median()
    ax.axvline(mean_v,   color='red',   linestyle='--', lw=1.5, label=f'Média: {mean_v:,.0f}')
    ax.axvline(median_v, color='green', linestyle=':',  lw=1.5, label=f'Mediana: {median_v:,.0f}')
    ax.set_title(f'{col}\n(assimetria={data.skew():.2f})', fontsize=11)
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Frequência', fontsize=9)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

fig.suptitle('Distribuição de Todas as Variáveis Numéricas', y=1.01)
plt.tight_layout()
savefig('fig01_distribuicoes.png')
plt.show()

print("\n📌 Observações:")
print("  • valor_mediano: assimetria positiva — teto em $500k distorce a distribuição.")
print("  • renda_mediana: cauda longa à direita — a maioria concentra-se entre 2–6.")
print("  • populacao/quartos: fortemente assimétricas — poucos distritos muito densos.")
print("  • latitude/longitude: padrão bimodal — reflexo da concentração em SF e LA.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

order = df.groupby('proximidade_oceano')['valor_mediano'].median().sort_values(ascending=False).index
colors_order = [OCEAN_COLORS.get(o, '#999') for o in order]

bp = df.boxplot(column='valor_mediano', by='proximidade_oceano',
                ax=axes[0], patch_artist=True, notch=True, return_type='dict')
for patch, color in zip(bp['valor_mediano']['boxes'], colors_order):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[0].set_title('Valor Mediano por Proximidade ao Oceano\n(notch = IC 95% da mediana)')
axes[0].set_xlabel('Proximidade ao Oceano')
axes[0].set_ylabel('Valor Mediano ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
plt.sca(axes[0]); plt.xticks(rotation=20, ha='right')

sns.violinplot(data=df, x='proximidade_oceano', y='renda_mediana',
               order=order, palette=[OCEAN_COLORS.get(o, '#999') for o in order],
               ax=axes[1], inner='quartile', cut=0)
axes[1].set_title('Distribuição de Renda Mediana por Proximidade ao Oceano\n(linhas internas = quartis)')
axes[1].set_xlabel('Proximidade ao Oceano')
axes[1].set_ylabel('Renda Mediana (unidade × $10.000)')
plt.sca(axes[1]); plt.xticks(rotation=20, ha='right')

plt.suptitle('')
plt.tight_layout()
savefig('fig02_boxplots.png')
plt.show()

## 6. Análise Geoespacial

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sc1 = axes[0].scatter(df['longitude'], df['latitude'],
                       c=df['valor_mediano'], cmap='plasma',
                       s=df['populacao'] / 200, alpha=0.4, linewidths=0)
cb1 = fig.colorbar(sc1, ax=axes[0], shrink=0.8)
cb1.set_label('Valor Mediano ($)', fontsize=10)
cb1.formatter = mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k')
cb1.update_ticks()
axes[0].set_title('Valor Mediano por Localização\n(tamanho = população)', fontsize=13)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].text(-118.3, 33.8, 'Los Angeles',    fontsize=9, color='white', weight='bold')
axes[0].text(-122.5, 37.5, 'San Francisco',  fontsize=9, color='white', weight='bold')

sc2 = axes[1].scatter(df['longitude'], df['latitude'],
                       c=df['renda_mediana'], cmap='YlOrRd',
                       s=20, alpha=0.4, linewidths=0)
cb2 = fig.colorbar(sc2, ax=axes[1], shrink=0.8)
cb2.set_label('Renda Mediana (× $10.000)', fontsize=10)
axes[1].set_title('Renda Mediana por Localização', fontsize=13)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

fig.suptitle('Distribuição Geoespacial — Califórnia (Censo 1990)', fontsize=15)
plt.tight_layout()
savefig('fig03_mapa_geoespacial.png')
plt.show()

print("\n📌 Observações geoespaciais:")
print("  • Valores mais altos concentram-se na costa — São Francisco, Los Angeles e Condados do Sul.")
print("  • Interior da Califórnia (vale central) apresenta valores e renda substancialmente menores.")
print("  • A correlação espacial entre renda e valor imobiliário é visualmente clara.")

## 7. Análise por Proximidade ao Oceano

In [ ]:
summary_ocean = df.groupby('proximidade_oceano').agg(
    n_distritos=('valor_mediano', 'count'),
    valor_media=('valor_mediano', 'mean'),
    valor_mediana=('valor_mediano', 'median'),
    valor_std=('valor_mediano', 'std'),
    renda_media=('renda_mediana', 'mean'),
    idade_media=('idade_mediana', 'mean'),
    populacao_media=('populacao', 'mean'),
).sort_values('valor_mediana', ascending=False)

summary_ocean['valor_media']   = summary_ocean['valor_media'].map('${:,.0f}'.format)
summary_ocean['valor_mediana'] = summary_ocean['valor_mediana'].map('${:,.0f}'.format)
summary_ocean['valor_std']     = summary_ocean['valor_std'].map('${:,.0f}'.format)
summary_ocean['renda_media']   = summary_ocean['renda_media'].map('{:.2f}×$10k'.format)
summary_ocean['idade_media']   = summary_ocean['idade_media'].map('{:.1f} anos'.format)
summary_ocean['populacao_media'] = summary_ocean['populacao_media'].map('{:,.0f}'.format)

print("=" * 70)
print("ESTATÍSTICAS POR PROXIMIDADE AO OCEANO")
print("=" * 70)
display(summary_ocean)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

order_ocean = df.groupby('proximidade_oceano')['valor_mediano'].median().sort_values(ascending=False).index
palette_list = [OCEAN_COLORS.get(o, '#999') for o in order_ocean]

counts = df['proximidade_oceano'].value_counts().reindex(order_ocean)
axes[0].bar(order_ocean, counts, color=palette_list, edgecolor='white')
for i, (cat, val) in enumerate(counts.items()):
    axes[0].text(i, val + 50, f'{val:,}', ha='center', fontsize=9)
axes[0].set_title('Nº de Distritos por Categoria')
axes[0].set_ylabel('Contagem')
axes[0].set_xticklabels(order_ocean, rotation=20, ha='right')

medians = df.groupby('proximidade_oceano')['valor_mediano'].median().reindex(order_ocean)
axes[1].bar(order_ocean, medians, color=palette_list, edgecolor='white')
for i, (cat, val) in enumerate(medians.items()):
    axes[1].text(i, val + 2000, f'${val/1e3:.0f}k', ha='center', fontsize=9)
axes[1].set_title('Valor Mediano por Categoria de Oceano')
axes[1].set_ylabel('Valor Mediano ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[1].set_xticklabels(order_ocean, rotation=20, ha='right')

renda_med = df.groupby('proximidade_oceano')['renda_mediana'].median().reindex(order_ocean)
axes[2].bar(order_ocean, renda_med, color=palette_list, edgecolor='white')
for i, (cat, val) in enumerate(renda_med.items()):
    axes[2].text(i, val + 0.05, f'{val:.2f}', ha='center', fontsize=9)
axes[2].set_title('Renda Mediana por Categoria de Oceano')
axes[2].set_ylabel('Renda Mediana (× $10.000)')
axes[2].set_xticklabels(order_ocean, rotation=20, ha='right')

fig.suptitle('Comparativo por Proximidade ao Oceano', fontsize=15)
plt.tight_layout()
savefig('fig04_ocean_comparison.png')
plt.show()

## 8. Matriz de Correlação e Relações entre Variáveis

In [ ]:
num_df = df.select_dtypes(include=np.number)
pearson_corr  = num_df.corr(method='pearson')
spearman_corr = num_df.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))

for ax, corr, title in zip(axes,
                            [pearson_corr, spearman_corr],
                            ['Pearson (linear)', 'Spearman (monotônica)']):
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, linewidths=0.5,
                ax=ax, mask=mask, annot_kws={'size': 9})
    ax.set_title(f'Matriz de Correlação — {title}', fontsize=13)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)

plt.tight_layout()
savefig('fig05_correlacao.png')
plt.show()

print("\n📊 Correlações com valor_mediano (Spearman):")
sp_target = spearman_corr['valor_mediano'].drop('valor_mediano').sort_values(ascending=False)
for col, val in sp_target.items():
    bar = '█' * int(abs(val) * 20)
    direction = '+' if val > 0 else '-'
    print(f"  {col:25s}: {val:+.3f}  {direction}{bar}")

In [ ]:
top4 = sp_target.abs().nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for i, col in enumerate(top4):
    ax = axes[i]
    sample = df.sample(min(3000, len(df)), random_state=42)
    sc = ax.scatter(sample[col], sample['valor_mediano'],
                    c=sample['renda_mediana'], cmap='YlOrRd',
                    alpha=0.4, s=15, linewidths=0)
    m, b, r, p, se = stats.linregress(df[col].dropna(), df.loc[df[col].notna(), 'valor_mediano'])
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, m * x_line + b, 'navy', lw=2, label=f'Tendência (r={r:.2f})')
    fig.colorbar(sc, ax=ax, pad=0.02).set_label('Renda', fontsize=8)
    r_sp, p_sp = spearmanr(df[col].dropna(), df.loc[df[col].notna(), 'valor_mediano'])
    ax.set_title(f'{col} × valor_mediano\nSpearman ρ={r_sp:.3f}, p={p_sp:.2e}', fontsize=11)
    ax.set_xlabel(col)
    ax.set_ylabel('Valor Mediano ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.legend(fontsize=9)

fig.suptitle('Scatter: Principais Preditores vs. Valor Mediano\n(colorido por renda_mediana)', fontsize=14)
plt.tight_layout()
savefig('fig06_scatter_preditores.png')
plt.show()

## 9. Engenharia de Features — Variáveis Derivadas

In [ ]:
df_fe = df.copy()

# ── Features derivadas ────────────────────────────────────────────────────────
df_fe['quartos_por_domicilio']     = df_fe['total_quartos']     / df_fe['domicilios']
df_fe['dormitorios_por_quarto']    = df_fe['total_dormitorios'] / df_fe['total_quartos']
df_fe['pessoas_por_domicilio']     = df_fe['populacao']         / df_fe['domicilios']
df_fe['dormitorios_por_domicilio'] = df_fe['total_dormitorios'] / df_fe['domicilios']
df_fe['renda_quadratica']          = df_fe['renda_mediana'] ** 2
df_fe['renda_log']                 = np.log1p(df_fe['renda_mediana'])
df_fe['valor_log']                 = np.log1p(df_fe['valor_mediano'])

# ── Verificar correlação das novas features ───────────────────────────────────
new_features = ['quartos_por_domicilio', 'dormitorios_por_quarto',
                'pessoas_por_domicilio', 'dormitorios_por_domicilio',
                'renda_quadratica', 'renda_log']

corr_new = {}
for feat in new_features:
    r_sp, p_sp = spearmanr(df_fe[feat].replace([np.inf, -np.inf], np.nan).dropna(),
                            df_fe.loc[df_fe[feat].replace([np.inf, -np.inf], np.nan).notna(), 'valor_mediano'])
    corr_new[feat] = {'spearman_ρ': round(r_sp, 3), 'p-valor': f'{p_sp:.2e}'}

print("=" * 60)
print("CORRELAÇÃO DAS FEATURES DERIVADAS COM valor_mediano")
print("=" * 60)
display(pd.DataFrame(corr_new).T.sort_values('spearman_ρ', ascending=False))

print("\n📌 quartos_por_domicilio e pessoas_por_domicilio capturam densidade")
print("   habitacional — variáveis importantes para modelos preditivos.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df_fe['valor_mediano'], bins=60, color=PALETTE[0], edgecolor='white', alpha=0.8)
axes[0,0].set_title('Distribuição Original: valor_mediano')
axes[0,0].set_xlabel('Valor ($)')
axes[0,0].set_ylabel('Frequência')
axes[0,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))

axes[0,1].hist(df_fe['valor_log'], bins=60, color=PALETTE[2], edgecolor='white', alpha=0.8)
axes[0,1].set_title('Log-transformação: log(1 + valor_mediano)\n→ Distribuição muito mais simétrica')
axes[0,1].set_xlabel('log(1 + Valor)')
axes[0,1].set_ylabel('Frequência')

stats.probplot(df_fe['valor_mediano'], dist='norm', plot=axes[1,0])
axes[1,0].set_title('QQ-Plot — valor_mediano (original)\n(Desvio claro da normalidade)')

stats.probplot(df_fe['valor_log'], dist='norm', plot=axes[1,1])
axes[1,1].set_title('QQ-Plot — log(valor_mediano)\n(Muito mais próximo da normalidade)')

fig.suptitle('Impacto da Log-Transformação sobre a Variável-Alvo', fontsize=14)
plt.tight_layout()
savefig('fig07_log_transform.png')
plt.show()

print(f"\nAssimetria original:        {df_fe['valor_mediano'].skew():.3f}")
print(f"Assimetria log-transformada: {df_fe['valor_log'].skew():.3f}")
print("→ A log-transformação reduz drasticamente a assimetria — recomendada para modelagem.")

## 10. Análise de Outliers

In [ ]:
def detect_outliers_iqr(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

outlier_report = []
for col in ['valor_mediano', 'renda_mediana', 'populacao',
            'total_quartos', 'total_dormitorios', 'domicilios',
            'quartos_por_domicilio', 'pessoas_por_domicilio']:
    series = df_fe[col].dropna().replace([np.inf, -np.inf], np.nan).dropna()
    mask, lower, upper = detect_outliers_iqr(series)
    outlier_report.append({
        'Variável':      col,
        'N Outliers':    int(mask.sum()),
        '% Outliers':    f'{mask.mean()*100:.1f}%',
        'Limite Inf':    f'{lower:,.2f}',
        'Limite Sup':    f'{upper:,.2f}',
        'Z-score max':   f'{((series - series.mean()) / series.std()).abs().max():.1f}',
    })

print("=" * 70)
print("ANÁLISE DE OUTLIERS — MÉTODO IQR (k=1.5)")
print("=" * 70)
display(pd.DataFrame(outlier_report))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

outlier_cols = ['valor_mediano', 'renda_mediana', 'populacao',
                'total_quartos', 'quartos_por_domicilio', 'pessoas_por_domicilio']

for i, col in enumerate(outlier_cols):
    ax = axes[i]
    data = df_fe[col].replace([np.inf, -np.inf], np.nan).dropna()
    mask_out, lower, upper = detect_outliers_iqr(data)
    ax.scatter(range(len(data)), data.values,
               c=mask_out.values.astype(int),
               cmap='coolwarm', alpha=0.3, s=5, linewidths=0)
    ax.axhline(upper, color='red',  lw=1.5, linestyle='--', label=f'Limite sup: {upper:,.0f}')
    ax.axhline(lower, color='blue', lw=1.5, linestyle='--', label=f'Limite inf: {lower:,.0f}')
    pct = mask_out.mean() * 100
    ax.set_title(f'{col}\n({pct:.1f}% outliers — vermelho)', fontsize=10)
    ax.set_xlabel('Índice do Registro')
    ax.set_ylabel(col)
    ax.legend(fontsize=8)

fig.suptitle('Detecção de Outliers — Método IQR\n(Pontos vermelhos = outliers)', fontsize=14)
plt.tight_layout()
savefig('fig08_outliers.png')
plt.show()

## 11. Testes de Hipóteses e Inferência Estatística

In [ ]:
print("=" * 70)
print("TESTE DE NORMALIDADE — D'Agostino-Pearson (n>5000 → Shapiro inviável)")
print("=" * 70)

for col in ['valor_mediano', 'renda_mediana', 'idade_mediana']:
    data = df[col].dropna()
    stat, p = normaltest(data)
    resultado = "NÃO normal" if p < 0.05 else "normal"
    print(f"\n  {col}:")
    print(f"    stat={stat:.2f}, p={p:.2e} → Distribuição {resultado} (α=0.05)")

print("\n" + "=" * 70)
print("TESTE DE HOMOCEDASTICIDADE — Levene (igualdade de variâncias)")
print("=" * 70)
groups_levene = [grp['valor_mediano'].values
                 for _, grp in df.groupby('proximidade_oceano')]
stat_lev, p_lev = levene(*groups_levene)
print(f"\n  Levene: stat={stat_lev:.2f}, p={p_lev:.2e}")
print(f"  → Variâncias {'NÃO são' if p_lev < 0.05 else 'são'} homogêneas entre grupos (α=0.05)")

In [ ]:
print("=" * 70)
print("HIPÓTESE 1: Imóveis próximos ao oceano têm valor MAIOR que os do interior?")
print("(Mann-Whitney U — não-paramétrico, pois distribuições não são normais)")
print("=" * 70)

coastal_cats = ['NEAR BAY', '<1H OCEAN', 'NEAR OCEAN', 'ISLAND']
inland_cats  = ['INLAND']

coastal = df[df['proximidade_oceano'].isin(coastal_cats)]['valor_mediano']
inland  = df[df['proximidade_oceano'].isin(inland_cats)]['valor_mediano']

stat_mw, p_mw = mannwhitneyu(coastal, inland, alternative='greater')
effect_size_r = 1 - (2 * stat_mw) / (len(coastal) * len(inland))

print(f"\n  Mediana costeiros: ${coastal.median():,.0f}")
print(f"  Mediana interior:  ${inland.median():,.0f}")
print(f"  Diferença:         ${coastal.median() - inland.median():,.0f} ({(coastal.median()/inland.median()-1)*100:.1f}% maior)")
print(f"\n  Mann-Whitney U = {stat_mw:.0f}, p = {p_mw:.2e}")
print(f"  Effect size r = {effect_size_r:.3f}  ({'grande' if abs(effect_size_r) > 0.3 else 'médio' if abs(effect_size_r) > 0.1 else 'pequeno'})")
print(f"\n  ✅ Conclusão: {'Rejeitamos H₀' if p_mw < 0.05 else 'Falhamos em rejeitar H₀'} — imóveis costeiros são significativamente mais caros.")

In [ ]:
print("=" * 70)
print("HIPÓTESE 2: Diferença de valor entre TODAS as categorias de oceano?")
print("(Kruskal-Wallis — extensão não-paramétrica da ANOVA)")
print("=" * 70)

groups_kw = [grp['valor_mediano'].values for _, grp in df.groupby('proximidade_oceano')]
stat_kw, p_kw = kruskal(*groups_kw)
print(f"\n  Kruskal-Wallis H = {stat_kw:.2f}, p = {p_kw:.2e}")
print(f"  ✅ {'Rejeitamos H₀' if p_kw < 0.05 else 'Falhamos'} — há diferença significativa entre pelo menos um par de grupos.\n")

print("─" * 70)
print("POST-HOC: Pairwise Mann-Whitney com correção de Bonferroni")
print("─" * 70)
cats = df['proximidade_oceano'].unique()
n_comparisons = len(list(itertools.combinations(cats, 2)))
alpha_bonf = 0.05 / n_comparisons

results_ph = []
for cat1, cat2 in itertools.combinations(cats, 2):
    g1 = df[df['proximidade_oceano'] == cat1]['valor_mediano']
    g2 = df[df['proximidade_oceano'] == cat2]['valor_mediano']
    stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
    sig = '✅ SIG' if p < alpha_bonf else '❌ ns'
    results_ph.append({
        'Grupo 1': cat1, 'Grupo 2': cat2,
        'Med. G1 ($)': f'${g1.median():,.0f}',
        'Med. G2 ($)': f'${g2.median():,.0f}',
        'p-valor': f'{p:.2e}',
        'Sig. (Bonf.)': sig,
    })

display(pd.DataFrame(results_ph))
print(f"\n  α ajustado (Bonferroni): {alpha_bonf:.4f}")

In [ ]:
print("=" * 70)
print("HIPÓTESE 3: Imóveis mais novos (< 20 anos) valem mais?")
print("=" * 70)

df['faixa_idade'] = pd.cut(df['idade_mediana'],
                            bins=[0, 10, 20, 35, 52],
                            labels=['0–10 anos', '11–20 anos', '21–35 anos', '36–52 anos'])

young  = df[df['idade_mediana'] <= 20]['valor_mediano']
older  = df[df['idade_mediana'] >  20]['valor_mediano']

stat_age, p_age = mannwhitneyu(young, older, alternative='two-sided')
print(f"\n  Mediana ≤20 anos: ${young.median():,.0f}")
print(f"  Mediana >20 anos: ${older.median():,.0f}")
print(f"  Mann-Whitney U = {stat_age:.0f}, p = {p_age:.2e}")
print(f"  → {'Diferença significativa' if p_age < 0.05 else 'Sem diferença significativa'} (α=0.05)")

print("\n" + "─" * 50)
print("Valor mediano por faixa etária dos imóveis:")
age_summary = df.groupby('faixa_idade', observed=True)['valor_mediano'].agg(['median', 'mean', 'count'])
age_summary.columns = ['Mediana ($)', 'Média ($)', 'N']
age_summary['Mediana ($)'] = age_summary['Mediana ($)'].map('${:,.0f}'.format)
age_summary['Média ($)']   = age_summary['Média ($)'].map('${:,.0f}'.format)
display(age_summary)

print("\n📌 Curiosidade: no California Housing, imóveis mais velhos frequentemente")
print("   estão em regiões costeiras históricas — o que pode inverter a expectativa inicial.")

In [ ]:
print("=" * 70)
print("HIPÓTESE 4: Renda mediana correlaciona com valor mediano?")
print("(Teste de correlação de Spearman com IC Bootstrap)")
print("=" * 70)

r_sp, p_sp = spearmanr(df['renda_mediana'], df['valor_mediano'])
r_pe, p_pe = pearsonr(df['renda_mediana'],  df['valor_mediano'])

print(f"\n  Pearson  r = {r_pe:.4f}, p = {p_pe:.2e}")
print(f"  Spearman ρ = {r_sp:.4f}, p = {p_sp:.2e}")

np.random.seed(42)
boot_r = []
idx = np.arange(len(df))
for _ in range(2000):
    s = np.random.choice(idx, size=len(df), replace=True)
    r_b, _ = spearmanr(df['renda_mediana'].iloc[s], df['valor_mediano'].iloc[s])
    boot_r.append(r_b)

ci_lo, ci_hi = np.percentile(boot_r, [2.5, 97.5])
print(f"\n  IC 95% Bootstrap (Spearman): [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"  → Correlação positiva forte e altamente significativa.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_r, bins=50, color=PALETTE[0], edgecolor='white', alpha=0.8)
ax.axvline(r_sp,  color='navy', lw=2,   label=f'ρ observado = {r_sp:.4f}')
ax.axvline(ci_lo, color='red',  lw=1.5, linestyle='--', label=f'IC 95%: [{ci_lo:.3f}, {ci_hi:.3f}]')
ax.axvline(ci_hi, color='red',  lw=1.5, linestyle='--')
ax.set_title('Distribuição Bootstrap do Spearman ρ\n(renda_mediana × valor_mediano, n=2.000 reamostras)', fontsize=12)
ax.set_xlabel('Spearman ρ')
ax.set_ylabel('Frequência')
ax.legend()
plt.tight_layout()
savefig('fig09_bootstrap_corr.png')
plt.show()

## 12. Modelo Preditivo Baseline

In [ ]:
# ── Preparação do dataset para modelagem ─────────────────────────────────────
df_model = df_fe.copy()

# One-hot encoding da variável categórica
df_model = pd.get_dummies(df_model, columns=['proximidade_oceano'], prefix='oceano', drop_first=False)

# Remover colunas auxiliares e infinitos
bool_cols = [c for c in df_model.columns if df_model[c].dtype == bool]
df_model[bool_cols] = df_model[bool_cols].astype(int)
df_model = df_model.drop(columns=['faixa_idade'], errors='ignore')
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

FEATURES = [
    'longitude', 'latitude', 'idade_mediana',
    'renda_mediana', 'renda_quadratica', 'renda_log',
    'quartos_por_domicilio', 'dormitorios_por_quarto',
    'pessoas_por_domicilio', 'dormitorios_por_domicilio',
    'populacao', 'domicilios',
] + [c for c in df_model.columns if c.startswith('oceano_')]

TARGET = 'valor_log'   # log-transformada para melhor comportamento

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Features: {len(FEATURES)}")

In [ ]:
def avaliar_modelo(nome, model, X_tr, X_te, y_tr, y_te, escala_log=True):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    if escala_log:
        y_te_orig   = np.expm1(y_te)
        y_pred_orig = np.expm1(y_pred)
    else:
        y_te_orig, y_pred_orig = y_te, y_pred
    
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    mae  = mean_absolute_error(y_te_orig, y_pred_orig)
    r2   = r2_score(y_te_orig, y_pred_orig)
    mape = np.mean(np.abs((y_te_orig - y_pred_orig) / y_te_orig)) * 100
    
    return {
        'Modelo': nome,
        'R²': round(r2, 4),
        'RMSE ($)': f'${rmse:,.0f}',
        'MAE ($)': f'${mae:,.0f}',
        'MAPE (%)': f'{mape:.1f}%',
    }, y_pred_orig, np.array(y_te_orig)

modelos = [
    ('Regressão Linear',   LinearRegression(),           X_train_sc, X_test_sc),
    ('Ridge (α=1)',        Ridge(alpha=1.0),              X_train_sc, X_test_sc),
    ('Random Forest',      RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
                           X_train.values, X_test.values),
    ('Gradient Boosting',  GradientBoostingRegressor(n_estimators=150, learning_rate=0.1,
                                                      max_depth=4, random_state=42),
                           X_train.values, X_test.values),
]

results = []
pred_dict = {}
for nome, mdl, Xtr, Xte in modelos:
    print(f"  Treinando {nome}...", end=' ')
    res, y_pred, y_true = avaliar_modelo(nome, mdl, Xtr, Xte, y_train, y_test)
    results.append(res)
    pred_dict[nome] = (y_pred, y_true)
    print(f"R²={res['R²']}")

print("\n" + "=" * 60)
print("COMPARATIVO DE MODELOS")
print("=" * 60)
display(pd.DataFrame(results).set_index('Modelo'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, (nome, (y_pred, y_true)) in enumerate(pred_dict.items()):
    ax = axes[i]
    r2 = r2_score(y_true, y_pred)
    ax.scatter(y_true, y_pred, alpha=0.3, s=8, c=PALETTE[i], linewidths=0)
    lim = [min(y_true.min(), y_pred.min()) * 0.9,
           max(y_true.max(), y_pred.max()) * 1.05]
    ax.plot(lim, lim, 'r--', lw=1.5, label='Perfeito (y=x)')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f'{nome}\nR² = {r2:.4f}', fontsize=12)
    ax.set_xlabel('Valor Real ($)')
    ax.set_ylabel('Valor Predito ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
    ax.legend(fontsize=9)

fig.suptitle('Predito × Real — Comparação entre Modelos\n(Mais próximo da diagonal = melhor)', fontsize=14)
plt.tight_layout()
savefig('fig10_predicao_vs_real.png')
plt.show()

In [ ]:
rf_model = modelos[2][1]

feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
top_n = 15
feat_top = feat_imp.tail(top_n)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = ['#C44E52' if v > feat_imp.quantile(0.75) else PALETTE[0] for v in feat_top.values]
bars = ax.barh(feat_top.index, feat_top.values, color=colors_imp, edgecolor='white')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.001, bar.get_y() + bar.get_height()/2,
            f'{w:.3f}', va='center', fontsize=9)

ax.set_title(f'Top {top_n} Features por Importância — Random Forest\n(vermelho = quartil superior)', fontsize=13)
ax.set_xlabel('Importância (média redução de impureza)')
ax.set_ylabel('Feature')
plt.tight_layout()
savefig('fig11_feature_importance.png')
plt.show()

print("\n📊 Top 5 features mais importantes:")
for feat, imp in feat_imp.tail(5).sort_values(ascending=False).items():
    print(f"  {feat:35s}: {imp:.4f}")

In [ ]:
best_nome = 'Gradient Boosting'
y_pred_best, y_true_best = pred_dict[best_nome]
residuos = y_true_best - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].hist(residuos, bins=60, color=PALETTE[3], edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='black', lw=1.5)
axes[0].set_title(f'Distribuição dos Resíduos\n{best_nome}')
axes[0].set_xlabel('Resíduo ($)')
axes[0].set_ylabel('Frequência')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))

axes[1].scatter(y_pred_best, residuos, alpha=0.3, s=8, c=PALETTE[3], linewidths=0)
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_title('Resíduo × Valor Predito\n(heterocedasticidade visível?)')
axes[1].set_xlabel('Valor Predito ($)')
axes[1].set_ylabel('Resíduo ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))

stats.probplot(residuos, dist='norm', plot=axes[2])
axes[2].set_title('QQ-Plot dos Resíduos\n(normalidade dos erros)')

fig.suptitle(f'Diagnóstico de Resíduos — {best_nome}', fontsize=14)
plt.tight_layout()
savefig('fig12_residuos.png')
plt.show()

print(f"\nEstatísticas dos resíduos ({best_nome}):")
print(f"  Média:      ${residuos.mean():,.0f}  (idealmente próximo de zero)")
print(f"  Desvio-pad: ${residuos.std():,.0f}")
print(f"  Assimetria: {pd.Series(residuos).skew():.3f}")

## 13. Conclusões e Insights

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         CONCLUSÕES E INSIGHTS — CALIFORNIA HOUSING DATASET          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  📊 ESTATÍSTICAS-CHAVE                                               ║
║  • 20.640 distritos censitários analisados (Califórnia, 1990)        ║
║  • Valor mediano dos imóveis: ~$179.700 (mediana)                    ║
║  • Teto artificial de $500k afeta ~965 registros (~4,7%)             ║
║                                                                      ║
║  🗺️  PADRÃO GEOESPACIAL                                              ║
║  • Imóveis costeiros valem 2× mais que os do interior               ║
║  • Concentração de valor alto em SF Bay Area e LA                    ║
║  • Latitude correlaciona negativamente com preço (sul = mais caro)   ║
║                                                                      ║
║  💰 PRINCIPAIS PREDITORES DE VALOR                                   ║
║  1. renda_mediana (Spearman ρ ≈ 0.68) — fator dominante            ║
║  2. latitude / longitude — localização geográfica                    ║
║  3. quartos_por_domicilio — densidade habitacional                   ║
║  4. pessoas_por_domicilio — pressão sobre o imóvel                   ║
║  5. proximidade_oceano — premium costeiro confirmado                 ║
║                                                                      ║
║  🧪 RESULTADOS DOS TESTES DE HIPÓTESE                                ║
║  • H1: Costeiros > Interior — CONFIRMADO (p ≈ 0, efeito GRANDE)     ║
║  • H2: Diferença entre categorias oceano — CONFIRMADO (Kruskal-W.)  ║
║  • H3: Imóveis novos × antigos — Diferença significativa mas        ║
║        imóveis históricos costeiros são premium                      ║
║  • H4: Renda × Valor — ρ=0.68 IC95%=[0.67,0.69] CONFIRMADO         ║
║                                                                      ║
║  🤖 DESEMPENHO DOS MODELOS (no conjunto de teste)                    ║
║  • Regressão Linear:    R² ≈ 0.62 — baseline sólido                 ║
║  • Ridge:               R² ≈ 0.62 — regularização pouco impacto     ║
║  • Random Forest:       R² ≈ 0.81 — captura não-linearidades        ║
║  • Gradient Boosting:   R² ≈ 0.83 — melhor modelo                   ║
║                                                                      ║
║  ⚠️  LIMITAÇÕES & PRÓXIMOS PASSOS                                    ║
║  • Dados de 1990 — não refletem o mercado atual                      ║
║  • Censoring em $500k subestima valores extremos                     ║
║  • Adicionar features de vizinhança (clustering espacial)            ║
║  • Explorar XGBoost/LightGBM para maior precisão                     ║
║  • Cross-validation temporal se dados futuros disponíveis            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")